# Level 2 — Lesson 2: A First AI Model

**AI for Radiotherapy**

**Audience:** You completed Lesson 1 or already understand the dataset/problem-definition concepts.

**Teaching time:** ~45–50 minutes  
**Core question:** *How does a neural network learn a representation that is useful for a prediction task?*

We use the same OrganAMNIST problem:

```text
X = 28 × 28 preprocessed image
y = one of 11 organ classes
```

The goal is **not** benchmark performance. The goal is to understand:

- batches and epochs;
- model parameters;
- logits and probabilities;
- cross-entropy loss;
- loss versus metrics;
- gradients and optimizer updates;
- training versus validation;
- generalization and overfitting;
- learned feature representations.

# 0. Setup

In [1]:
!pip -q install medmnist==3.0.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.0 MB/s eta 0:00:00


In [2]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import medmnist
from medmnist import INFO

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [3]:
DATA_FLAG = "organamnist"
IMAGE_SIZE = 28

info = INFO[DATA_FLAG]
DataClass = getattr(medmnist, info["python_class"])
class_names = [info["label"][str(i)] for i in range(len(info["label"]))]

transform = transforms.ToTensor()

train_dataset = DataClass(split="train", transform=transform, download=True, size=IMAGE_SIZE)
val_dataset = DataClass(split="val", transform=transform, download=True, size=IMAGE_SIZE)
test_dataset = DataClass(split="test", transform=transform, download=True, size=IMAGE_SIZE)

train_labels = np.asarray(train_dataset.labels).squeeze()

print(
    f"{len(train_dataset):,} train | "
    f"{len(val_dataset):,} validation | "
    f"{len(test_dataset):,} test"
)
print(f"{len(class_names)} classes")

100%|██████████| 38.2M/38.2M [00:27<00:00, 1.38MB/s]


34,561 train | 6,491 validation | 17,778 test
11 classes


# 1. The learning loop

A supervised model can be summarized as:

```text
x
↓
model f(x, θ)
↓
prediction
↓
compare with y
↓
loss
↓
gradients
↓
update θ
```

Here:

- \(X\) = image;
- \(y\) = correct organ class;
- \(f_\theta\) = neural network;
- \(\theta\) = trainable parameters.

The **architecture** defines the form of \(f_\theta\).  
The **parameters** are the numerical weights and biases that training changes.  
The **hyperparameters** are all those model parameters that are not trained by gradient descent (e.g. size of hidden layers, loss parameters...).

# 2. Batches, iterations, epochs

A `DataLoader` groups examples into **minibatches**. Processing one minibatch, loss backpropagation and doing an optimizer step is an **iteration**. One complete pass through the training dataset is an **epoch**.

In [4]:
BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available()
)

images, targets = next(iter(train_loader))
targets = targets.squeeze(1).long()

print("Images:", images.shape)
print("Targets:", targets.shape)

Images: torch.Size([128, 1, 28, 28])
Targets: torch.Size([128])


Interpret `[128, 1, 28, 28]` as 128 examples, one channel, 28×28 pixels.

Later, 3D medical imaging typically uses:

```text
[batch, channel, x, y, z]
```

# What is a channel?

A **channel** is a separate feature map defined over the same spatial grid.

A familiar example is a colour image. An RGB image contains three channels:

```text
red
green
blue
```

Each channel has the same width and height, but stores different information at every pixel.

A neural network therefore sees an RGB image as:

```text
[channel, height, width]

[3, H, W]
```

For a typical radiotherapy CT, only one channel is required:

```text
CT
↓
[1, X, Y, Z]
```

The CT intensity values themselves contain the image information.

### Multiple channels can provide different types of spatial information

For some tasks, we may want to provide the model with additional information that is spatially aligned with the CT.

For example, segmentation masks can be supplied as separate channels:

```text
Channel 0: CT
Channel 1: PTV mask
Channel 2: spinal cord mask
Channel 3: left parotid mask
Channel 4: right parotid mask
```

Each segmentation channel could contain:

```text
0 = outside the structure
1 = inside the structure
```

At every voxel, the model therefore receives several pieces of information:

```text
CT intensity
PTV membership
spinal cord membership
parotid membership
...
```

Because all channels share the same spatial grid, the network can learn relationships between them.

For example, it might learn that a particular pattern depends on:

- distance from the PTV;
- proximity to an organ at risk;
- surrounding tissue density;
- combinations of anatomical structures.

This is one reason channels are particularly useful in radiotherapy AI: they allow us to represent several spatially related sources of information in a computationally convenient form.

### Exercise — design the input for dose prediction

Suppose we want to predict a 3D radiotherapy dose distribution.

**What information would you provide to the model as input?**

Possible candidates include:

```text
CT
PTV mask
OAR masks
prescription information
beam information
treatment technique
```

Now ask a more important question:

> **Which of these can naturally be represented as spatial channels, and which require a different representation?**

# 3. A small convolutional neural network

In the following we will design a small **convolutional neural network (CNN)** to classify the MedMNIST data we are already familiar with from lesson 1.

Think of the CNN as containing essentially these steps:

```text
image
↓
feature extractor
↓
learned feature vector
↓
classifier
↓
11 class scores
```

In [5]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        self.feature_extractor = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.classifier = nn.Linear(64, n_classes)

    def forward_features(self, x):
        x = self.feature_extractor(x)
        return torch.flatten(x, 1)

    def forward(self, x):
        features = self.forward_features(x)
        return self.classifier(features)

model = SmallCNN(n_classes=len(class_names)).to(device)

n_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"Trainable parameters: {n_parameters:,}")

SmallCNN(
  (feature_extractor): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (classifier): Linear(in_features=64, out_features=11, bias=True)
)
Trainable parameters: 24,011


##How is a PyTorch model defined?

In PyTorch, a neural-network architecture is usually defined as a Python class that inherits from nn.Module. Here, our architecture is called SmallCNN.

There are two important parts:

class SmallCNN(nn.Module):

    def __init__(self):
        ...
        
    def forward(self, x):
        ...

When the model is initialized, `__init__()` creates the layers that belong to the model.

Layers such as

`nn.Conv2d(...)`  
`nn.Linear(...)`

contain trainable parameters. They must be created once and stored as part of the model so that PyTorch can keep track of their weights and update them during training.

In our example, we group several operations using:

`nn.Sequential(...)`

`nn.Sequential` creates a pipeline that applies each operation to the input in the order in which it was specified:

input
  ↓
Conv2d
  ↓
ReLU
  ↓
MaxPool
  ↓
Conv2d
  ↓
...

The `forward()` function describes what happens when an input is passed through the network.

With `nn.Sequential`, this can be very compact, however, you may also encounter code bases where every layer is written out explicitly.

An equivalent architecture could be defined like this:

class SmallCNNExplicit(nn.Module):

    def __init__(self, n_classes):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.conv3(x)
        x = self.relu(x)

        x = self.global_pool(x)
        x = torch.flatten(x, 1)

        return self.classifier(x)

Both styles are common. The explicit form can be useful when the data flow is complicated or when intermediate outputs are needed. nn.Sequential is convenient when layers are simply applied one after another, because it keeps the code shorter and easier to read.

**Note**: we created the layers in `__init__()` and stored them as part of the current class instance `self`. If we had called the functions directly in `forward()`, the model would not be trainable. Make sure you understand why.

# 4. Logits → probabilities → loss

Before we look at training, lets first analyze what the model actually outputs.

The model produces 11 real-valued **logits**. These can be turned into "probabilities" using softmax:

```text
logits
↓
softmax
↓
probability for each class
```

Let's inspect our **untrained** model.

In [11]:
images, targets = next(iter(train_loader))
images = images.to(device)
targets = targets.squeeze(1).long().to(device)

with torch.no_grad():
    logits = model(images)
    probabilities = torch.softmax(logits, dim=1)

example = 0
true_class = targets[example].item()

probability_table = pd.DataFrame({
    "class": class_names,
    "logit": logits[example].cpu().numpy(),
    "probability": probabilities[example].cpu().numpy(),
})

display(probability_table)
most_likely_class = probability_table.loc[probability_table["probability"].idxmax(), "class"]

print("True class:", class_names[true_class])
print("Probability assigned to true class",
      f"{probabilities[example, true_class]:.3f}")
print(
    "Most likely class according to model:",
    f"{most_likely_class}"
)

,class,logit,probability
0,bladder,-0.133126,0.081243
1,femur-left,-0.072351,0.086334
2,femur-right,-0.103463,0.083689
3,heart,0.063250,0.098871
4,kidney-left,-0.078767,0.085781
5,kidney-right,-0.058760,0.087515
6,liver,-0.009690,0.091916
7,lung-left,0.034552,0.096074
8,lung-right,0.046777,0.097256
9,pancreas,0.002993,0.093090


True class: kidney-right
Probability assigned to true class 0.088
Most likely class according to model: heart


## Cross-entropy for one example

If the true class is \(y\), and the model assigns it probability \(p_y\), the cross entropy loss is

\
$L=-\log(p_y)$

Lower is better. Confidently wrong predictions are penalized strongly.


In [12]:
example_probabilities = np.array([0.99, 0.90, 0.50, 0.10, 0.01])

pd.DataFrame({
    "p(correct class)": example_probabilities,
    "-log(p)": -np.log(example_probabilities),
})

,p(correct class),-log(p)
0,0.99,0.010050
1,0.90,0.105361
2,0.50,0.693147
3,0.10,2.302585
4,0.01,4.605170


## CrossEntropyLoss, probabilities, and interpretation

Our neural network outputs logits: arbitrary real-valued scores for each class. PyTorch's CrossEntropyLoss accepts logits directly. Internally, it performs the equivalent of a log-softmax followed by the negative log-likelihood of the correct class. We therefore do not apply softmax ourselves before passing the output to CrossEntropyLoss.

After applying softmax, the resulting values can be interpreted as the probabilities that the model assigns to each class.

However, these values should not automatically be interpreted as reliable measures of uncertainty.

Deep neural networks are often poorly calibrated: the confidence expressed by their predicted probabilities does not necessarily correspond to how often those predictions are actually correct. In particular, neural networks can be overconfident, assigning very high probability to predictions that turn out to be wrong.

> **A high predicted probability means that the model strongly prefers that answer, but not that the answer is necessarily correct.**

For example, a model might predict:

liver:   0.98
spleen:  0.01
kidney:  0.01

and still be wrong.

If reliable probabilities or uncertainty estimates are important to the application, model calibration becomes an additional part of the machine-learning workflow.

A useful introduction to this topic is:

Guo et al. (2017), On Calibration of Modern Neural Networks
https://arxiv.org/abs/1706.04599

# 5. Put the loss on a meaningful scale

For (\$K$\) equally likely classes, a uniform predictor has a cross-entropy loss of

\
$L_{\text{uniform}} = \log(K)$


For 11 classes this is about 2.40.

Because the dataset is not perfectly balanced, an image-blind predictor could instead always output the observed class-frequency distribution. If (\$q_k$\)
is the probability of class k the expected cross-entropy is

\
$-\sum_k q_k \log(q_k).$


For **accuracy**, a separate simple baseline is to always choose the most common class.

Loss and accuracy answer different questions.

In [13]:
counts = pd.Series(train_labels).value_counts().sort_index()
class_probabilities = counts.values / counts.values.sum()

uniform_ce = np.log(len(class_names))
frequency_ce = -np.sum(class_probabilities * np.log(class_probabilities))
majority_accuracy = class_probabilities.max()

print(f"Uniform 11-class CE baseline: {uniform_ce:.3f}")
print(f"Class-frequency CE baseline:  {frequency_ce:.3f}")
print(f"Majority-class accuracy:      {majority_accuracy:.3f}")

Uniform 11-class CE baseline: 2.398
Class-frequency CE baseline:  2.294
Majority-class accuracy:      0.178


## Loss is what we optimize; a metric is what we use to judge performance

```text
LOSS
chosen partly because it is suitable for optimization

METRIC
chosen because it measures behaviour we care about
```

Here:

```text
loss   = cross-entropy
metric = accuracy
```

Later this distinction becomes even more important:

```text
segmentation:
    training loss → e.g. Dice + cross-entropy
    evaluation    → Dice + surface metrics

dose prediction:
    training loss → e.g. voxel-wise dose error
    evaluation    → dose error + DVH-based metrics
```

# 6. What actually changes during learning?

We have built out model, inspected its output, discussed a loss function. Next up is training. Lets build a one-step training to first understand what happens in this process, before we dive fully into a training loop.

The most notable new addition is the `optimizer`. Here we use `ADAM`, the most common optimizer in medical AI, and likely overall.

In [14]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

images, targets = next(iter(train_loader))
images = images.to(device)
targets = targets.squeeze(1).long().to(device)

weights_before = model.feature_extractor[0].weight.detach().clone()

optimizer.zero_grad()
logits = model(images)
loss = loss_function(logits, targets)
loss.backward()
optimizer.step()

weights_after = model.feature_extractor[0].weight.detach().clone()

change = torch.linalg.vector_norm(weights_after - weights_before).item()

print(f"Batch loss: {loss.item():.3f}")
print(f"Change in first-layer weights: {change:.6f}")

Batch loss: 2.389
Change in first-layer weights: 0.011983


A training step contains distinct operations:

```python
optimizer.zero_grad()
logits = model(images)
loss = loss_function(logits, targets)
loss.backward()
optimizer.step()
```

- forward pass → compute prediction;
- loss → quantify error;
- `backward()` → calculate gradients;
- `step()` → change parameters;
- `zero_grad()` → clear old gradients.

### Predict before trying

What happens if we remove `optimizer.step()`?  
What happens if we remove `loss.backward()`?  
What might a learning rate of `10` do?

# 7. Training and generalization

Now that we understood the general training idea, lets start an actual training. Here is where the split into training and validation set we did at the beginning is important.

There are two different questions:

> **Optimization:** Can I reduce the loss on the training data?

> **Generalization:** Does what I learned work on independent data?

The validation set helps us estimate generalization during development. The test set should remain untouched until development decisions are finished.

The following example shows you how a typical PyTorch training loop looks like.

In [1]:
def run_epoch(model, loader, loss_function, optimizer=None):
    training = optimizer is not None

    model.train() if training else model.eval()

    total_loss = 0.0
    n_correct = 0
    n_examples = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.squeeze(1).long().to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = loss_function(logits, targets)

            if training:
                loss.backward()
                optimizer.step()

        predictions = logits.argmax(dim=1)
        batch_size = images.shape[0]

        total_loss += loss.item() * batch_size
        n_correct += (predictions == targets).sum().item()
        n_examples += batch_size

    return total_loss / n_examples, n_correct / n_examples

In [16]:
set_seed()

model = SmallCNN(n_classes=len(class_names)).to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

N_EPOCHS = 5

history = {
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": [],
}

for epoch in range(1, N_EPOCHS + 1):
    train_loss, train_acc = run_epoch(
        model, train_loader, loss_function, optimizer
    )
    val_loss, val_acc = run_epoch(
        model, val_loader, loss_function
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_accuracy"].append(train_acc)
    history["val_accuracy"].append(val_acc)

    print(
        f"Epoch {epoch:02d}/{N_EPOCHS} | "
        f"train loss {train_loss:.3f} | val loss {val_loss:.3f} | "
        f"train acc {train_acc:.3f} | val acc {val_acc:.3f}"
    )

Epoch 01/5 | train loss 1.647 | val loss 1.226 | train acc 0.385 | val acc 0.545
Epoch 02/5 | train loss 1.109 | val loss 1.008 | train acc 0.588 | val acc 0.626
Epoch 03/5 | train loss 0.901 | val loss 0.640 | train acc 0.680 | val acc 0.807
Epoch 04/5 | train loss 0.740 | val loss 0.714 | train acc 0.744 | val acc 0.747
Epoch 05/5 | train loss 0.662 | val loss 0.481 | train acc 0.773 | val acc 0.851


In [2]:
epochs = np.arange(1, N_EPOCHS + 1)

plt.figure(figsize=(7, 4))
plt.plot(epochs, history["train_loss"], marker="o", label="training")
plt.plot(epochs, history["val_loss"], marker="o", label="validation")
plt.axhline(uniform_ce, linestyle="--",
            label=f"uniform baseline ({uniform_ce:.2f})")
plt.axhline(frequency_ce, linestyle=":",
            label=f"frequency baseline ({frequency_ce:.2f})")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Loss relative to simple baselines")
plt.legend()
plt.show()

NameError: name 'np' is not defined

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(epochs, history["train_accuracy"], marker="o", label="training")
plt.plot(epochs, history["val_accuracy"], marker="o", label="validation")
plt.axhline(majority_accuracy, linestyle="--",
            label=f"majority baseline ({majority_accuracy:.2f})")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.title("Accuracy during training")
plt.legend()
plt.show()

### Read loss and accuracy together

```text
loss ↓, accuracy similar
```
The model may be assigning more probability to predictions that were already correct.

```text
accuracy ↑, loss ↑
```
Some decisions may improve while a smaller number of confidently wrong predictions become worse.

```text
training loss ↓, validation loss ↑
```
A classic sign of overfitting.

We rarely rely on one number, and defining the best metric for evaluating/quantifying a task matters a lot. In radiotherapy, the quantity of interest often differs from typical machine learning metrics. For example, we are interested in dosimetric quality of treatment plans. A segmentation's dice score or Hausdorff-distance does not tell us whether:

- Segmentation errors cause overdose in OARs or target underdose in the plan
- Dosimetric differences arising from the segemntation are clinically meaningful

There are full papers just about metric definition, see, e.g., Maier-Hein, L., Reinke, A., Godau, P. et al. Metrics reloaded: recommendations for image analysis validation. Nat Methods 21, 195–212 (2024). https://doi.org/10.1038/s41592-023-02151-z

## Overfitting

Go back to the training section and increase the number of epochs. For speed, you may want to run this experiment using a GPU.

> **What happens to the training and validation loss as training continues?**

Training a neural network is an optimization problem: we adjust the model parameters to reduce the loss on the **training data**.

If our only objective were to minimize the training loss, we could continue until that loss no longer improves.

But that is not our actual goal.

Our goal is to build a model that **generalizes to unseen data**.

As training continues, two things may happen:

```text
training loss ↓
validation loss ↓
```

The model is learning patterns that also generalize to unseen data.

Later, however, we may observe:

```text
training loss ↓
validation loss ↑
```

The model is still becoming better at fitting the training data, but its performance on unseen data is getting worse.

This is **overfitting**.

The model has started to learn patterns that are specific to the training data rather than patterns that generalize reliably to new examples.

At the other extreme, we have **underfitting**:

```text
training loss high
validation loss high
```

Here the model has not captured the relevant structure of the training data sufficiently well.

A useful mental picture is therefore:

```text
                 useful training
                      ↓
underfitting ─────────┼───────── overfitting
                      ↑
              best generalization
```

The goal of model training is **not to obtain the smallest possible training loss**.

It is to find model parameters that perform well on data that were not used to fit those parameters.

This is one reason we monitor a separate **validation set** during training.

A common strategy is **early stopping**:

> Stop training when the validation performance stops improving, even if the training loss could continue to decrease.

Importantly, training for many epochs does not automatically cause overfitting. Whether overfitting occurs depends on the model capacity, amount and diversity of training data, regularization, augmentation, and the learning problem itself.

> **Optimization tells us how well we can fit the training data. Generalization tells us whether what we learned is useful beyond it.**

# 8. One final test-set evaluation

In [ ]:
test_loss, test_accuracy = run_epoch(model, test_loader, loss_function)

print(f"Test loss:     {test_loss:.3f}")
print(f"Test accuracy: {test_accuracy:.3f}")

If we repeatedly inspect test performance and alter the model in response, the test set has effectively become another validation set.

## Briefly inspect where errors occur

In [ ]:
def collect_predictions(model, loader):
    y_true, y_pred = [], []
    model.eval()

    with torch.no_grad():
        for images, targets in loader:
            logits = model(images.to(device))
            y_pred.append(logits.argmax(dim=1).cpu().numpy())
            y_true.append(targets.squeeze(1).numpy())

    return np.concatenate(y_true), np.concatenate(y_pred)

y_true, y_pred = collect_predictions(model, test_loader)

cm = confusion_matrix(
    y_true, y_pred, labels=np.arange(len(class_names))
)

fig = plt.figure(figsize=(9, 9))
ax = fig.add_axes([0.12, 0.18, 0.72, 0.72])
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(ax=ax, xticks_rotation=60, values_format="d")
ax.set_title("Test-set confusion matrix")
plt.show()

The point here is not to perform a full evaluation. It is simply to see that the same overall accuracy can hide very different class-specific error patterns.

# 9. The model learned a new representation

In Lesson 1, we analyzed the 784-dimensional datapoints with a PCA:

```text
image → 784 raw pixel features
```

The CNN extracts features into a 64-dimensional feature vector. This is called **embedding** or **representation**.

```text
image
↓
feature extractor
↓
64 learned features
↓
classifier
```

Training encouraged those 64 features to be useful for distinguishing the target classes.

We now project the **raw input representation** and the **learned representation** into 2D using the same PCA idea.

In [ ]:
N_EMBED = min(3000, len(test_dataset))
rng = np.random.default_rng(SEED)
embed_indices = rng.choice(len(test_dataset), size=N_EMBED, replace=False)

raw_test = test_dataset.imgs[embed_indices].astype(np.float32) / 255.0
embed_labels = np.asarray(test_dataset.labels)[embed_indices].squeeze()
X_raw = raw_test.reshape(N_EMBED, -1)

embed_loader = DataLoader(
    Subset(test_dataset, embed_indices),
    batch_size=256,
    shuffle=False,
    num_workers=2,
)

feature_batches = []
label_batches = []

model.eval()
with torch.no_grad():
    for images, targets in embed_loader:
        features = model.forward_features(images.to(device))
        feature_batches.append(features.cpu().numpy())
        label_batches.append(targets.squeeze(1).numpy())

X_learned = np.concatenate(feature_batches)
feature_labels = np.concatenate(label_batches)

print("Raw representation:", X_raw.shape)
print("Learned representation:", X_learned.shape)

In [ ]:
raw_pca = PCA(n_components=2, random_state=SEED)
X_raw_2d = raw_pca.fit_transform(X_raw)

plt.figure(figsize=(8, 6))
for class_id, class_name in enumerate(class_names):
    mask = embed_labels == class_id
    plt.scatter(
        X_raw_2d[mask, 0], X_raw_2d[mask, 1],
        s=9, alpha=0.55, label=class_name
    )
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.title("Raw representation: 784 pixels → 2D PCA")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
feature_pca = PCA(n_components=2, random_state=SEED)
X_learned_2d = feature_pca.fit_transform(X_learned)

plt.figure(figsize=(8, 6))
for class_id, class_name in enumerate(class_names):
    mask = feature_labels == class_id
    plt.scatter(
        X_learned_2d[mask, 0], X_learned_2d[mask, 1],
        s=9, alpha=0.55, label=class_name
    )
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.title("Learned representation: 64 CNN features → 2D PCA")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## What changed?

Do not compare literal coordinates between the two plots; they are separate PCA spaces.

Ask instead:

- Are classes more clearly organized in the learned feature space?
- Which classes remain difficult?
- What objective encouraged the representation to change?

This is **representation learning**.

Be precise:

> The model has not necessarily discovered "true anatomy", causal mechanisms, or clinically meaningful biomarkers. It has learned a representation useful for the objective, data, and distribution we supplied.

# 10. Optional extension — UMAP of the learned features

If you used UMAP in Lesson 1, apply the same visualization to the learned features.

The learning objective is not "how to use UMAP"; it is:

> We can inspect high-dimensional representations using a human-readable projection while remembering that the projection can distort structure.

In [ ]:
# OPTIONAL
!pip -q install umap-learn

In [ ]:
# OPTIONAL
import umap.umap_ as umap

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=20,
    min_dist=0.15,
    metric="euclidean",
    random_state=SEED,
)

X_learned_umap = reducer.fit_transform(X_learned)

plt.figure(figsize=(8, 6))
for class_id, class_name in enumerate(class_names):
    mask = feature_labels == class_id
    plt.scatter(
        X_learned_umap[mask, 0], X_learned_umap[mask, 1],
        s=9, alpha=0.55, label=class_name
    )
plt.xlabel("UMAP dimension 1")
plt.ylabel("UMAP dimension 2")
plt.title("Optional: UMAP of the learned CNN representation")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

# 11. Exit check

Explain without looking at the notebook:

1. What are \(X\) and \(y\)?
2. What is the difference between architecture and parameters?
3. What are logits?
4. Why is uniform 11-class cross-entropy about 2.40?
5. Why are loss and accuracy not interchangeable?
6. What does `loss.backward()` do?
7. What does `optimizer.step()` do?
8. Why do we need validation data?
9. What is overfitting?
10. What is a learned representation?

# 12. Bridge to Lesson 3: MONAI

The core learning loop is now familiar:

```python
prediction = model(x)
loss = loss_function(prediction, y)

optimizer.zero_grad()
loss.backward()
optimizer.step()
```

MONAI helps us build the **medical-imaging machinery around it**:

```text
medical images / labels
↓
loading + metadata
↓
transforms
↓
sampling + augmentation
↓
tensors
↓
PyTorch model and training loop
```

The next question is:

> **How should medical-imaging data be presented to the model?**

---

## References

- MedMNIST: https://medmnist.com/
- MedMNIST repository and metadata: https://github.com/MedMNIST/MedMNIST
- Yang J, Shi R, Wei D, et al. *MedMNIST v2: A Large-Scale Lightweight Benchmark for 2D and 3D Biomedical Image Classification.* Scientific Data. 2023.
- OrganAMNIST is distributed by MedMNIST under CC BY 4.0.

This notebook is educational material and is not intended for clinical use.